# 00 · Thiết lập dùng chung

Mọi notebook gọi `%run ./00_setup.ipynb` ở đầu. Cả ba cell đều ẨN.

### 00-01 · Mọi notebook dùng chung đường dẫn, seed và bảng ánh xạ cột
**ẨN**

Mục tiêu: import, seed, đường dẫn, bảng ánh xạ COL sang tên cột thật của dataset.

In [1]:
# 00-01 · ẨN
import os, sys, json, warnings
from pathlib import Path

# Output gọn trên lớp: tắt thanh tiến trình và log tải mô hình.
for key, value in {"TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TRANSFORMERS_VERBOSITY": "error", "HF_HUB_VERBOSITY": "error", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(key, value)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import *  # COL, đường dẫn, cấu hình
from src.common import set_seed, versions, cached, show, Timer, fmt_vnd

set_seed(SEED)
pd.set_option("display.max_colwidth", 80)
VERSIONS = versions()  # ghi phiên bản thư viện để tái lập

### 00-02 · Cấu hình LLM, embedding và cơ sở dữ liệu nằm một chỗ
**ẨN**

Mục tiêu: đọc cấu hình từ biến môi trường; khoá API không ghi trong notebook.

In [2]:
# 00-02 · ẨN
# Khoá Claude: đặt ANTHROPIC_API_KEY trong môi trường hoặc chạy `ant auth login` trước khi mở Jupyter.
CONFIG = {
    "llm": LLM_MODEL, "effort": LLM_EFFORT, "embedding": EMBEDDING_MODEL, "rerank": RERANK_MODEL,
    "vector_db": f"{QDRANT_URL} ({QDRANT_COLLECTION})", "graph_db": NEO4J_URI,
    "dataset": DATASET, "n_rows": N_ROWS, "chunk": (CHUNK_SIZE, CHUNK_OVERLAP), "rebuild": REBUILD,
}
for folder in (DATA_DIR, INDEX_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

### 00-03 · Hàm dùng chung nằm trong src/, notebook chỉ gọi
**ẨN**

Mục tiêu: nạp các module theo hợp đồng ở mục 2.3.

In [3]:
# 00-03 · ẨN
from src import data, vector, qdrant_store, rag, graph, hybrid
from src import eval as ev
from src.llm import ask_llm
from src.rag import search_dense, search_bm25, rrf, ask_rag
from src.graph import run_cypher, ask_graph
from src.hybrid import router, ask_hybrid